# Watering reminder demo

This shows a per-plant watering series and the difference between postponing one reminder and moving the whole series; previously the project could record completed watering only. Schedules are stored separately from completed events so a reminder never claims that watering happened. The common clock time is configurable, while a one-off postponement leaves the anchor date intact. Plant history and care advice were deliberately left unchanged.

The input plant comes from the public fixture in `data/demo_garden.json`. The demo creates an isolated schema in `TEST_DATABASE_URL`, uses no Telegram credentials, and removes its data afterward.

In [1]:
import json
import os
from datetime import UTC, date, datetime, time
from pathlib import Path
from uuid import UUID, uuid4

import psycopg
from psycopg import sql
from psycopg.conninfo import make_conninfo

from her_garden.models import PlantState
from her_garden.store import GardenStore
from her_garden.watering import WateringStore

fixture = json.loads(Path("data/demo_garden.json").read_text())
print("Public example plant:", fixture["plant"]["name"])

Public example plant: Crassula


In [2]:
async def show_schedule() -> None:
    dsn = os.environ["TEST_DATABASE_URL"]
    schema = "demo_" + uuid4().hex
    async with await psycopg.AsyncConnection.connect(dsn, autocommit=True) as conn:
        await conn.execute(sql.SQL("CREATE SCHEMA {}").format(sql.Identifier(schema)))
    garden = GardenStore(make_conninfo(dsn, options=f"-csearch_path={schema}"))
    await garden.open()
    try:
        plant = await garden.create_plant(uuid4(), PlantState(**fixture["plant"]))
        plant_id = UUID(plant["entity_id"])
        watering = WateringStore(garden, "UTC")
        now = datetime(2026, 9, 23, 8, tzinfo=UTC)
        plan = await watering.set_schedule(uuid4(), plant_id, date(2026, 9, 23), 3, now)
        print("Original:", plan["anchor_date"], plan["next_due_at"])
        once = await watering.adjust_schedule(uuid4(), plant_id, 1, "once", now)
        print("One reminder +1 day:", once["anchor_date"], once["next_due_at"])
        series = await watering.adjust_schedule(uuid4(), plant_id, 2, "series", now)
        print("Whole series +2 days:", series["anchor_date"], series["next_due_at"])
        await watering.set_reminder_time(uuid4(), time(10, 30), now)
        updated = await watering.get_schedule(plant_id)
        print("Common time changed:", updated["next_due_at"])
        context = await garden.get_plant_context(plant_id)
        print("Completed watering events:", context["latest_actions"])
    finally:
        await garden.close()
        async with await psycopg.AsyncConnection.connect(dsn, autocommit=True) as conn:
            await conn.execute(sql.SQL("DROP SCHEMA {} CASCADE").format(sql.Identifier(schema)))


await show_schedule()

Original: 2026-09-23 2026-09-23T09:00:00+00:00
One reminder +1 day: 2026-09-23 2026-09-24T09:00:00+00:00
Whole series +2 days: 2026-09-25 2026-09-28T09:00:00+00:00
Common time changed: 2026-09-28T10:30:00+00:00
Completed watering events: {}
